In [1]:
pip install ptflops

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install -U wandb

In [3]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install thop

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms

class CIFAR10Custom(Dataset):
    def __init__(self, train=True):
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616]
            )
        ])

        self.dataset = torchvision.datasets.CIFAR10(
            root="./data",
            train=train,
            download=True,
            transform=self.transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, label


def get_dataloader(batch_size=128, train=True):
    dataset = CIFAR10Custom(train=train)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        num_workers=2,
        pin_memory=True
    )


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 8x8
        x = self.pool(F.relu(self.conv3(x)))   # 4x4

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


In [6]:
import wandb
import torch
import numpy as np

def log_gradient_flow(model):
    grad_norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norms[name] = param.grad.norm().item()
    wandb.log({"Gradient Flow": grad_norms})


def log_weight_flow(model):
    weight_norms = {}
    for name, param in model.named_parameters():
        weight_norms[name] = param.data.norm().item()
    wandb.log({"Weight Flow": weight_norms})


In [8]:
from kaggle_secrets import UserSecretsClient
import os
import wandb

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: b23cm1007 (b23cm1007-indian-institute-of-technology-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import wandb

# ------------------ CONFIG ------------------
EPOCHS = 30
BATCH_SIZE = 128
LR = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# --------------------------------------------

wandb.init(
    project="cifar10-cnn-lab2",
    name="SimpleCNN-CIFAR10",
    config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "model": "SimpleCNN"
    }
)

train_loader = get_dataloader(BATCH_SIZE, train=True)
test_loader = get_dataloader(BATCH_SIZE, train=False)

model = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

def train():
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch [{epoch+1}/{EPOCHS}]",
            leave=False
        )

        for images, labels in progress_bar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100 * correct / total:.2f}%"
            })

        log_gradient_flow(model)
        log_weight_flow(model)

        acc = 100 * correct / total
        wandb.log({
            "Train Loss": running_loss / len(train_loader),
            "Train Accuracy": acc,
            "Epoch": epoch
        })

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {running_loss:.4f}, Acc: {acc:.2f}%"
        )

train()
wandb.finish()

100%|██████████| 170M/170M [00:02<00:00, 79.0MB/s] 


Epoch [1/30] Loss: 618.7302, Acc: 42.09%


Epoch [2/30] Loss: 471.2358, Acc: 56.72%


Epoch [3/30] Loss: 404.8308, Acc: 63.15%


Epoch [4/30] Loss: 358.6283, Acc: 67.43%


...

Epoch [27/30] Loss: 169.3516, Acc: 84.87%


Epoch [28/30] Loss: 164.6576, Acc: 85.12%


Epoch [29/30] Loss: 163.3390, Acc: 85.20%


Epoch [30/30] Loss: 159.7263, Acc: 85.45%


Epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
Train Accuracy,▁▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████
Train Loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
Epoch,29
Train Accuracy,85.452
Train Loss,0.40851


In [13]:
from ptflops import get_model_complexity_info

with torch.cuda.device(0):
    macs, params = get_model_complexity_info(model, (3,32,32),
                                             as_strings=True,
                                             print_per_layer_stat=True)
    print(f"Total MACs: {macs}, Total Params: {params}")

SimpleCNN(
  620.36 k, 100.000% Params, 10.96 MMac, 98.962% MACs, 
  (conv1): Conv2d(896, 0.144% Params, 917.5 KMac, 8.282% MACs, 3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(18.5 k, 2.981% Params, 4.73 MMac, 42.740% MACs, 32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(73.86 k, 11.905% Params, 4.73 MMac, 42.666% MACs, 64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(0, 0.000% Params, 57.34 KMac, 0.518% MACs, kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(524.54 k, 84.555% Params, 524.54 KMac, 4.735% MACs, in_features=2048, out_features=256, bias=True)
  (fc2): Linear(2.57 k, 0.414% Params, 2.57 KMac, 0.023% MACs, in_features=256, out_features=10, bias=True)
)
Total MACs: 11.08 MMac, Total Params: 620.36 k
